# Erzeuge Embeddings für text Chunks
Die Chunks müssen vorher erstellt und in der Tabelle bge_m3_vectors gespeichert werden

## Environment

In [1]:
import os
os.environ["IND_PG_SCHEMA"] = "meipi-indexing"
os.environ["IND_DATA_DIR"] = "/home/padmin/Development/projekte/meipi-indexing/data"

## Import

In [2]:
import sqlalchemy as sa
from sqlalchemy.orm import aliased
from meipi.indexing import DBOperations, DBBgeM3Vector, DBMeta
from meipi.indexing.model import ChunkItem, DocItem
from meipi.indexing.config import EmbeddingConfig, PipelineConfig
#from meipi.indexing.embedding_pipeline import DocItem, EmbeddingPipeline
from meipi.indexing.pipeline import Batch, STOP, ThreadedEmbeddingPipeline


In [ ]:
pool_id = 1
numdocs = 10
V = aliased(DBMeta)
dbop = DBOperations(pool_id)
with dbop.Session() as session:
    res = session.execute(sa.select(V.id, V.inhalt).where(V.ftype == "doc").limit(numdocs)).fetchall()
    docs = [DocItem(**row._asdict()) for row in res]
print(len(docs))
print(docs[0])


100
DocItem(id=112152, inhalt='Putzplan 2601-04.odt\n\nJanuar - April 2026\n\nAnja\n\nvon bis\n\nDienstag, 27.1.\n\nDienstag, 24.2.\n\nDienstag, 24.3.\n\nDienstag, 21.4. D')


## Konfiguration

In [ ]:

qsize = 100
num_workers = 4
batch_size = 16
emb_config = EmbeddingConfig(num_workers=num_workers, batch_size =batch_size)
pipeline_config = PipelineConfig(max_queue_size=qsize, n_db_read_workers=2,
                                 n_chunking_workers=2, n_embedding_workers=1,
                                 n_postprocess_workers=2, n_db_write_workers=2)


## Run Pipeline

In [ ]:
id_list = [doc.id for doc in docs]
pipeline = ThreadedEmbeddingPipeline(emb_config, pipeline_config, pool_id)
res = pipeline.run_pipeline(id_list, test=True)

In [7]:
from pprint import pprint
print(len(res))
display(res[0])

18


DBBgeM3Vector(doc_id=117893, chunk_index=0, content='Registergericht Freiburg HRB 5718 Geschäftsführung: Tobias Lagatz, Christa van der Burgh Gläubiger-ID: DE68ZZZ00000207489 GLN-Nr.: 4024896000001 Die rechtsgeschäftliche Durchführung (Auslieferung, Abrechnung, etc.) erfolgt durch die Haufe Service Center GmbH im eigenen Namen für Rechnung Dritter (Kommission). Kommittenten sind u. a.: Haufe-Lexware GmbH & Co. KG, Haufe Akademie GmbH & Co. KG, Haufe-Lexware Real Estate AG, VCW Verlag für ControllingWissen AG, Schäffer-Poeschel GmbH. Haufe Service Center GmbH Munzinger Straße 9 79111 Freiburg Steuer-Nr.: 06430/43767 USt-IdNr.: DE 812499727 Bitte nicht überweisen. Ausstehende Beträge werden automatisch von Ihrem Konto oder Ihrer Kreditkarte abgebucht. www.lexoffice.de lexoffice@haufe-lexware.net Internet: E-Mail: Dr. Meinolf Piwek Meinolf Piwek Teylestr. 14 44791 Bochum Rechnung Rechnungsnummer lx2022050024558 Zahlungsziel 11.05.2022 Leistungszeitraum 04.05.2022 - 04.06.2022 Rechnungsdat

In [ ]:
pprint([(l.doc_id, l.chunk_index) for l in l])

## Test

In [ ]:
from sqlalchemy.sql import null


with dbop.Session() as session:
    stmt = sa.select(sa.func.count()).select_from(DBBgeM3Vector).where(DBBgeM3Vector.content == null())
    res = session.execute(stmt).scalar_one()
    print(res)


## New Pipeline

In [ ]:
x = DBBgeM3Vector(doc_id=1, chunk_index=1, content="test", vector=[1,2,3])
print(x)